# Data Lab 7 — Predicting Failures (scikit-learn)
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Split data into train and test sets.
- Train a classifier and read a confusion matrix / precision & recall.
- Spot the imbalanced-data accuracy trap and rank feature importance.

## Build features & target
Predict `fail` (1/0) from three process values. scikit-learn is pre-installed in Colab.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
df = pd.read_csv(DATA + "production_log.csv")
df["fail"] = (df["result"] == "Fail").astype(int)
X = df[["cycle_time_s", "oven_temp_c", "vibration_mm_s"]]
y = df["fail"]
print("features:", list(X.columns), "| overall fail rate:", round(y.mean(), 3))

## 1 · Train / test split
Fit on one part of the data, test on data the model has never seen.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape)

## 2 · Train a decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = DecisionTreeClassifier(max_depth=3, random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, pred), 3))

## 3 · The accuracy trap
~92% accuracy looks great — until you check how many actual failures it catches.

In [ ]:
print(confusion_matrix(y_test, pred), "\n")
print(classification_report(y_test, pred, target_names=["Pass", "Fail"], zero_division=0))

When failures are rare (~9%), a model can score high accuracy by mostly predicting "Pass" while **missing most failures**. Always read **recall** for the class you care about, not just accuracy.

## 4 · Rebalance to catch more failures
`class_weight="balanced"` tells the model failures matter more.

In [ ]:
model2 = DecisionTreeClassifier(max_depth=3, random_state=0, class_weight="balanced")
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, pred2), 3))
print(classification_report(y_test, pred2, target_names=["Pass", "Fail"], zero_division=0))

Recall for **Fail** goes up while overall accuracy dips — the **precision/recall trade-off**. Which matters more depends on the cost of a missed failure vs a false alarm.

## 5 · What drives the prediction?

In [ ]:
imp = pd.Series(model2.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
print(imp.round(3))

## Debrief
- Temperature should top the importance list — the same driver Lab 4 found by correlation, now confirmed by a model.
- This is **predictive analytics**: flag a likely failure *before* the part is scrapped.
- Caveat: this is a teaching model on synthetic data — real deployment needs validation, monitoring, and domain review.